In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
with open('transcript.txt') as fopen:
    d = fopen.read().split('\n')
d = [d_.split('|') for d_ in d if '|' in d_]
len(d)

6841

In [8]:
d[0][2]

'kono mae sagut ta toki wa 、 tochu- ni hankon no ryu-ki ga at ta node 、 tsui soko ga yukidomari da to bakari omot te 、 a- yut ta n desu ga 、'

In [6]:
!mkdir jss_audio

In [9]:
def loop(rows):
    rows, _ = rows
    data = []
    for r in tqdm(rows):
        try:
            f = r[0]
    
            audio_np, sr = sf.read(f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
    
            audio_filename = f.replace('/', '_').replace('.wav', '.mp3')
            audio_filename = os.path.join('jss_audio', audio_filename)
            
            t = r[1].strip()
            if len(t) < 2:
                continue
                
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': "JSS"
            })

            data.append({
                'audio_filename': audio_filename,
                'text': r[2].strip(),
                'speaker': "JSS"
            })

            
        except Exception as e:
            print(e)
            pass
            
    return data
            

In [11]:
data = loop((d[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 17.14it/s]


In [14]:
data[:2]

[{'audio_filename': 'jss_audio/meian_meian_0000.mp3',
  'text': 'この前探った時は、途中に瘢痕の隆起があったので、ついそこが行きどまりだとばかり思って、ああ云ったんですが、',
  'speaker': 'JSS'},
 {'audio_filename': 'jss_audio/meian_meian_0000.mp3',
  'text': 'kono mae sagut ta toki wa 、 tochu- ni hankon no ryu-ki ga at ta node 、 tsui soko ga yukidomari da to bakari omot te 、 a- yut ta n desu ga 、',
  'speaker': 'JSS'}]

In [16]:
data = multiprocessing(d, loop, cores = 20)

100%|██████████| 342/342 [00:21<00:00, 16.15it/s]


In [17]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'jss_audio/meian_meian_0000.mp3',
 'text': 'この前探った時は、途中に瘢痕の隆起があったので、ついそこが行きどまりだとばかり思って、ああ云ったんですが、',
 'speaker': 'JSS'}

In [18]:
dataset.push_to_hub('malaysia-ai/Japanese-Single-Speaker-TTS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 126.29ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  94%|█████████▍| 1.08MB / 1.14MB, 5.40MB/s  
Processing Files (1 / 1): 100%|██████████| 1.14MB / 1.14MB, 3.44MB/s  
Processing Files (1 / 1): 100%|██████████| 1.14MB / 1.14MB, 2.85MB/s  
New Data Upload: 100%|██████████| 1.14MB / 1.14MB, 2.85MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.21 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Japanese-Single-Speaker-TTS/commit/12f8094ee8b875b75cace899074f5cd66d028025', commit_message='Upload dataset', commit_description='', oid='12f8094ee8b875b75cace899074f5cd66d028025', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Japanese-Single-Speaker-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Japanese-Single-Speaker-TTS'), pr_revision=None, pr_num=None)

In [19]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'JSS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 128.23ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 1.14MB / 1.14MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  2.80 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/f80a55b14894097c7bdf9048cd01b80dcb1d469b', commit_message='Upload dataset', commit_description='', oid='f80a55b14894097c7bdf9048cd01b80dcb1d469b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [25]:
audio_files = [d['audio_filename'] for d in data]

with open('JSS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [22]:
# !zip -rq jss_audio.zip jss_audio

In [24]:
# !hf upload malaysia-ai/Japanese-Single-Speaker-TTS jss_audio.zip --repo-type=dataset

In [28]:
# !zip -rq jss_audio_neucodec.zip jss_audio_neucodec

In [29]:
# !hf upload malaysia-ai/Multilingual-TTS jss_audio_neucodec.zip --repo-type=dataset